# Drop Redundant Metric Layers & Save v4 Parquet

After the deep dive in notebook 04, we keep **only** the columns needed for modelling:

| Keep | Description |
|------|-------------|
| **Structural / ID** | `wy_player_id`, height, weight, age, transfer value/fee, minutes, position, season, team, comp metadata |
| **Twelve Quality Scores** (20 from + 20 to) | Composite metrics computed by Twelve Football |
| **Team stats** (from + to) | Team-level performance metrics |

**Dropped:**
- Other quality / ratio metrics (Wyscout-derived: Aerials won %, Shot conversion %, etc.)
- Per-90 counting stats
- Position-adjusted z-scores

These layers are redundant because the Twelve Quality Scores are already computed from z-scores, which in turn come from the per-90 stats. Keeping all layers would introduce multicollinearity.

**Input:** `within_league_transfers_v3.parquet`  
**Output:** `within_league_transfers_v4.parquet`

In [1]:
import pandas as pd
import os

INPUT_PATH  = "../../../thesis_data/processed_data/thesis_model_dataset/within_league_transfers_v3.parquet"
OUTPUT_PATH = "../../../thesis_data/processed_data/thesis_model_dataset/within_league_transfers_v4.parquet"

df = pd.read_parquet(INPUT_PATH)
print(f"Input: {df.shape[0]:,} rows x {df.shape[1]} columns")

Input: 18,065 rows x 512 columns


In [2]:
# --- Define the 20 Twelve Quality Score names ---
TWELVE_QS_NAMES = [
    "Active defence", "Aerial threat", "Box threat", "Chance prevention",
    "Composure", "Defensive heading", "Dribbling", "Effectiveness",
    "Finishing", "Hold-up play", "Intelligent defence", "Involvement",
    "Passing quality", "Poaching", "Pressing", "Progression",
    "Providing teammates", "Run quality", "Territorial dominance", "Winning duels",
]

from_twelve_qs = [f"from_{n}" for n in TWELVE_QS_NAMES]
to_twelve_qs   = [f"to_{n}" for n in TWELVE_QS_NAMES]

# --- Structural columns ---
STRUCTURAL = [
    "wy_player_id", "wyscout_height", "wyscout_weight", "player_season_age",
    "tm_transfer_value", "tm_transfer_fee",
    "from_Minutes", "from_competition", "from_position", "from_season", "from_team_id",
    "to_Minutes", "to_competition", "to_position", "to_season", "to_team_id",
    "from_comp_division", "from_comp_season_id", "from_comp_season_name",
    "to_comp_division", "to_comp_season_id", "to_comp_season_name",
]

# --- Team stats ---
from_team_stats = sorted([c for c in df.columns if c.startswith("from_team_stats_")])
to_team_stats   = sorted([c for c in df.columns if c.startswith("to_team_stats_")])

# --- Build keep list ---
KEEP = STRUCTURAL + from_twelve_qs + to_twelve_qs + from_team_stats + to_team_stats

# Verify all exist
missing = [c for c in KEEP if c not in df.columns]
if missing:
    print(f"WARNING — missing columns: {missing}")
else:
    print(f"All {len(KEEP)} columns to keep found in dataframe.")

All 210 columns to keep found in dataframe.


In [3]:
# --- What gets dropped ---
dropped = sorted(set(df.columns) - set(KEEP))

print(f"Columns KEPT:    {len(KEEP)}")
print(f"Columns DROPPED: {len(dropped)}")
print(f"\nDropped columns by type:")

drop_zscores = [c for c in dropped if "z_score" in c]
drop_per90   = [c for c in dropped if c.endswith("_per_90")]
drop_other   = [c for c in dropped if c not in drop_zscores and c not in drop_per90]

print(f"  Z-scores:              {len(drop_zscores)}")
print(f"  Per-90 stats:          {len(drop_per90)}")
print(f"  Other quality/ratio:   {len(drop_other)}")

if drop_other:
    print(f"\nOther quality/ratio columns dropped:")
    for c in drop_other:
        print(f"    {c}")

Columns KEPT:    210
Columns DROPPED: 302

Dropped columns by type:
  Z-scores:              150
  Per-90 stats:          0
  Other quality/ratio:   152

Other quality/ratio columns dropped:
    from_Aerials per 90
    from_Aerials won %
    from_Aerials won per 90
    from_Assists per 90
    from_Attacking aerials won %
    from_Attacking aerials won per 90
    from_Ball progression (xT) per 90
    from_Ball recoveries per 90
    from_Ball runs (xT) per 90
    from_Box entries per 90
    from_Carries (xT) per 100 receptions
    from_Carries (xT) per 90
    from_Counterpressing per 90
    from_Creative passes per 90
    from_Crosses (xT) per 90
    from_Deep completions per 90
    from_Deep runs (xT) per 90
    from_Defending 1v1 %
    from_Defensive actions per 90
    from_Defensive aerials won %
    from_Defensive aerials won per 90
    from_Defensive area (m^2)
    from_Defensive duels won %
    from_Defensive duels won per 90
    from_Defensive intensity per 90
    from_Defensive l

In [4]:
# --- Select & save ---
df_v4 = df[KEEP].copy()

print(f"Output: {df_v4.shape[0]:,} rows x {df_v4.shape[1]} columns")
print(f"\nColumn groups:")
print(f"  Structural:            {len(STRUCTURAL)}")
print(f"  Twelve QS (from):      {len(from_twelve_qs)}")
print(f"  Twelve QS (to):        {len(to_twelve_qs)}")
print(f"  Team stats (from):     {len(from_team_stats)}")
print(f"  Team stats (to):       {len(to_team_stats)}")
print(f"  TOTAL:                 {df_v4.shape[1]}")

Output: 18,065 rows x 210 columns

Column groups:
  Structural:            22
  Twelve QS (from):      20
  Twelve QS (to):        20
  Team stats (from):     74
  Team stats (to):       74
  TOTAL:                 210


In [5]:
# Quick sanity — null summary
null_pct = (100 * df_v4.isnull().sum() / len(df_v4)).round(1)
has_nulls = null_pct[null_pct > 0]

if len(has_nulls) > 0:
    print(f"Columns with nulls ({len(has_nulls)}):")
    print(has_nulls.sort_values(ascending=False).to_string())
else:
    print("No nulls in any column.")

Columns with nulls (182):
wyscout_height                                                              77.0
wyscout_weight                                                              77.0
from_Territorial dominance                                                  52.5
to_Territorial dominance                                                    52.5
from_Chance prevention                                                      52.5
to_Chance prevention                                                        52.5
from_Poaching                                                               48.0
to_Poaching                                                                 46.3
tm_transfer_fee                                                             35.6
to_comp_division                                                            29.7
to_comp_season_id                                                           29.7
from_comp_season_name                                                       29.7
fr

In [6]:
df_v4.to_parquet(OUTPUT_PATH, index=False)
size_mb = os.path.getsize(OUTPUT_PATH) / 1e6
print(f"Saved: {OUTPUT_PATH}")
print(f"Size:  {size_mb:.1f} MB")
print(f"Shape: {df_v4.shape[0]:,} rows x {df_v4.shape[1]} columns")

Saved: ../../../thesis_data/processed_data/thesis_model_dataset/within_league_transfers_v4.parquet
Size:  14.1 MB
Shape: 18,065 rows x 210 columns


In [7]:
# Final column listing
print("All columns in v4:")
for i, c in enumerate(df_v4.columns, 1):
    print(f"  {i:3d}. {c}")

All columns in v4:
    1. wy_player_id
    2. wyscout_height
    3. wyscout_weight
    4. player_season_age
    5. tm_transfer_value
    6. tm_transfer_fee
    7. from_Minutes
    8. from_competition
    9. from_position
   10. from_season
   11. from_team_id
   12. to_Minutes
   13. to_competition
   14. to_position
   15. to_season
   16. to_team_id
   17. from_comp_division
   18. from_comp_season_id
   19. from_comp_season_name
   20. to_comp_division
   21. to_comp_season_id
   22. to_comp_season_name
   23. from_Active defence
   24. from_Aerial threat
   25. from_Box threat
   26. from_Chance prevention
   27. from_Composure
   28. from_Defensive heading
   29. from_Dribbling
   30. from_Effectiveness
   31. from_Finishing
   32. from_Hold-up play
   33. from_Intelligent defence
   34. from_Involvement
   35. from_Passing quality
   36. from_Poaching
   37. from_Pressing
   38. from_Progression
   39. from_Providing teammates
   40. from_Run quality
   41. from_Territorial domin